# R2-3 / R3: Diagnosing the Table 1 recall anomalies

Reviewers 2 and 3 flagged two issues in the column-recall column of Table 1:

1. **Full-schema (C1) strict recall < 1.0** — impossible for a context that serializes the whole schema, unless the gold labels contain tokens absent from the schema.
2. **On BIRD, recall rises from L1 (0.868) to L2 (0.921)** although L2 prunes more — a non-monotonicity.

This notebook reproduces both from the released run and identifies the cause of each. It is read-only over `generation_results.csv`.

In [1]:
# Locate the master results table (released with the repo).
import os
from pathlib import Path
import pandas as pd, numpy as np

CANDIDATES = [os.environ.get("MASTER_CSV", ""),
              "results/tables/generation_results.csv",
              "../results/tables/generation_results.csv",
              "notebooks/results/tables/generation_results.csv"]
MASTER = next((p for p in CANDIDATES if p and Path(p).exists()), None)
assert MASTER, "Set MASTER_CSV or place generation_results.csv under results/tables/"
df = pd.read_csv(MASTER)
print("rows:", len(df), "| conditions:", sorted(df.condition.unique()))
print("tiers:", sorted(df.model_tier.unique()), "| datasets:", sorted(df.dataset.unique()))
OUT = Path("results"); (OUT/"tables").mkdir(parents=True, exist_ok=True); (OUT/"figures").mkdir(parents=True, exist_ok=True)


rows: 32598 | conditions: ['C1', 'C2', 'C3', 'C4', 'C5_L1', 'C5_L2', 'C5_L3']
tiers: ['MID', 'STRONG', 'WEAK'] | datasets: ['D1', 'D2', 'D2_fin']


## The paper's "column recall" is the strict metric
We confirm the Table 1 numbers correspond to `col_recall_strict` (model-independent, so it is identical across tiers).

In [2]:
s = df[df.model_tier=='STRONG']
rec = s.groupby(['dataset','condition'])[['col_recall_strict','col_recall_relaxed']].mean().round(4)
print(rec.to_string())
print('\nPaper Table 1 (BIRD col recall): C1=.979 C4=.970 L1=.868 L2=.921 L3=.876')
print('Paper Table 1 (Spider col recall): C1=.922 C4=.921 L1=.921 L2=.836 L3=.630')

                   col_recall_strict  col_recall_relaxed
dataset condition                                       
D1      C1                    0.9217              0.9298
        C2                    0.9206              0.9287
        C3                    0.8743              0.8852
        C4                    0.9212              0.9292
        C5_L1                 0.9208              0.9289
        C5_L2                 0.8360              0.8636
        C5_L3                 0.6303              0.6460
D2      C1                    0.9786              0.9861
        C2                    0.8086              0.8750
        C3                    0.9672              0.9800
        C4                    0.9704              0.9807
        C5_L1                 0.8680              0.9210
        C5_L2                 0.9211              0.9409
        C5_L3                 0.8761              0.8931
D2_fin  C1                    0.9792              0.9875
        C4                    0

## Issue 1 — C1 (full schema) strict recall < 1.0 is a gold-label artifact
Because C1 selects every column, strict recall below 1.0 can only mean the gold set holds tokens that do not exist anywhere in the schema. We measure how often, and whether even a relaxed (name-only) match fails.

In [3]:
for ds in ['D1','D2']:
    c1 = s[(s.dataset==ds)&(s.condition=='C1')]
    lt1 = c1[c1.col_recall_strict<0.999]
    print(f'{ds}: mean C1 strict recall={c1.col_recall_strict.mean():.4f}, '
          f'frac questions <1.0 = {(c1.col_recall_strict<0.999).mean():.3f}')
    print(f'    of those, relaxed (name-only) match ALSO fails on {(lt1.col_recall_relaxed<0.999).mean():.2f} '
          f'-> genuinely schema-absent tokens (SELECT *, aliases, derived exprs)')

D1: mean C1 strict recall=0.9217, frac questions <1.0 = 0.237
    of those, relaxed (name-only) match ALSO fails on 0.90 -> genuinely schema-absent tokens (SELECT *, aliases, derived exprs)
D2: mean C1 strict recall=0.9786, frac questions <1.0 = 0.070
    of those, relaxed (name-only) match ALSO fails on 0.77 -> genuinely schema-absent tokens (SELECT *, aliases, derived exprs)


**Conclusion (Issue 1):** C1 < 1.0 is not a retrieval failure; it is unresolvable gold tokens. Resolving gold columns against the real schema (patch `00b`) lifts C1 strict recall to 1.0 by construction.

## Issue 2 — BIRD L1 < L2 is a real non-monotonicity in the *old* level design
We check monotonicity per dataset and, on BIRD, count per-question how often L2 beats L1.

In [4]:
print('Monotonic check (strict recall; should be non-increasing C4>=L1>=L2>=L3):')
for ds in ['D1','D2']:
    r = s[s.dataset==ds].groupby('condition').col_recall_strict.mean()
    seq = [round(r[c],3) for c in ['C4','C5_L1','C5_L2','C5_L3']]
    tag = 'MONOTONE' if seq==sorted(seq,reverse=True) else '*** NON-MONOTONE ***'
    print(f'  {ds}: C4={seq[0]} L1={seq[1]} L2={seq[2]} L3={seq[3]}  {tag}')

d2 = s[s.dataset=='D2']
piv = d2.pivot_table(index='qid', columns='condition', values='col_recall_strict')
diff = piv['C5_L2'] - piv['C5_L1']
print(f'\nBIRD per-question: L2>L1 on {(diff>0).sum()} questions, L1>L2 on {(diff<0).sum()}, equal {(diff==0).sum()}')

Monotonic check (strict recall; should be non-increasing C4>=L1>=L2>=L3):
  D1: C4=0.921 L1=0.921 L2=0.836 L3=0.63  MONOTONE
  D2: C4=0.97 L1=0.868 L2=0.921 L3=0.876  *** NON-MONOTONE ***

BIRD per-question: L2>L1 on 143 questions, L1>L2 on 45, equal 312


**Conclusion (Issue 2):** On BIRD the old L1 and L2 used different column-selection mechanisms (L2 re-added embedding-preserved columns and all keys that L1 dropped), so the levels were not nested and recall was not monotone. The revised design (patch `01`) makes each level a strict subset of C4 (C4 ⊇ L1 ⊇ L2 ⊇ L3), which guarantees non-increasing recall. Both fixes are applied in the pipeline notebooks; this notebook documents the diagnosis on the original run.